# 📓 Semana 14 · Dia 2 — Primeiro agente LangGraph com ferramentas

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | GenAI Engineer Associate |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Agente respondendo com ferramentas |

---


## 📖 Teoria — Tool calling no LangGraph

O agente recebe a lista de **tools** (funções com docstring). O LLM decide chamar uma tool; o grafo executa e devolve o resultado; o LLM responde.

```
entrada → LLM → (chama tool?) → executa tool → LLM → resposta
```


### 💻 Na prática — Definindo as tools

Crie tools que consultam o Lakehouse.


In [ ]:
# Tools do agente de vendas
def receita_por_pais(pais: str) -> str:
    """Retorna a receita total de um país."""
    r = spark.sql(f"SELECT receita_total FROM workspace.ouro.receita_por_pais WHERE UPPER(Country) = UPPER('{pais}')").collect()
    return str(r[0][0]) if r else "País não encontrado."

def top_produtos(n: int = 5) -> str:
    """Retorna os n produtos mais vendidos."""
    rows = spark.sql(f"SELECT Description, receita_total FROM workspace.ouro.top_produtos ORDER BY receita_total DESC LIMIT {n}").collect()
    return "; ".join(f"{r[0]}: {r[1]}" for r in rows)
print("Tools definidas.")

In [ ]:
# Agente com tool calling (LangChain)
from langchain_community.chat_models import ChatDatabricks
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool

llm = ChatDatabricks(endpoint="databricks-llama-3-1-70b", temperature=0)
tools = [tool(receita_por_pais), tool(top_produtos)]
prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente de dados. Use as ferramentas disponíveis."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")])
agente = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agente, tools=tools, verbose=True)
print("Agente montado.")

In [ ]:
# Testar
resposta = executor.invoke({"input": "Qual a receita do United Kingdom?"})
print("Resposta:", resposta["output"])

### 💻 Na prática — Observando o loop

Com `verbose=True`, veja: o LLM decide chamar `receita_por_pais`, executa, observa o número e responde — o ciclo ReAct na prática.


> 🎯 **Dica de prova**: Agentes: tool calling = o LLM emite a chamada; o framework executa; o resultado volta como observação. Pergunta: 'como o agente decide qual tool?' → o LLM decide com base na descrição das tools.


## 🎯 Exercícios de fixação

**1.** Adicione uma tool `top_paises(n)`.

**2.** O que acontece se a tool falhar?

**3.** Por que a docstring da tool importa?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Tool nova

`def top_paises(n)` consultando receita_por_pais e retornando texto.

**2.** Tool falha

O erro vira observação; o LLM pode tentar outra tool ou responder com a limitação — por isso capture exceções.

**3.** Docstring

É a descrição que o LLM usa para DECIDIR qual tool chamar — docstring ruim = tool nunca chamada.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*